# Plot the three 42-bin mean-field parameter planes

This notebook reads only the newly calculated files in `data/recomputed_exact/`. It does not rerun the stable-root scans or pseudo-arclength continuation. Panels (a), (b), and (c) are generated independently as PNG and PDF files. The displayed state taxonomy is the three-state description: BP, BU, and NU, with coexistence regions retained.

## 0. Configuration

In [ ]:
from pathlib import Path
import importlib
import os
import sys

os.environ['B_CUTOFF'] = '0.56'
os.environ['BIMODALITY_MODE'] = 'sample_corrected'
os.environ['STATE_MODE'] = 'three'
os.environ['SEPARATE_C_NEAR_LINES'] = '1'

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'src' / 'mean_field_parameter_planes.py').is_file():
    raise RuntimeError('Run this notebook from paper-code-release/ or paper-code-release/notebooks/.')

data_dir = root / 'data' / 'recomputed_exact'
figure_dir = root / 'figures' / 'recomputed_exact_panels'
figure_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(root / 'src'))
sys.modules.pop('mean_field_parameter_planes', None)
mf = importlib.import_module('mean_field_parameter_planes')

# Redirect every panel input to the newly calculated data.
mf.PANEL_A_GRID = data_dir / 'mf_42bin_four_state_B056_sample_corrected_composition_grid.csv'
mf.PANEL_A_BOUNDARIES = data_dir / 'mf_42bin_four_state_B056_sample_corrected_continuous_boundaries.csv'
mf.PANEL_B_CACHE = data_dir / 'alpha2_dc_42bin_full_stability_step0p01.pkl'
mf.PANEL_B_CURVES = data_dir / 'alpha2_dc_42bin_true_pseudoarclength_curves.csv'
mf.PANEL_B_MACRO = data_dir / 'alpha2_dc_42bin_macroscopic_boundaries.csv'
mf.PANEL_B_PITCHFORK = data_dir / 'alpha2_dc_42bin_pitchfork_criticality_fine.csv'
mf.PANEL_B_PC = data_dir / 'alpha2_dc_42bin_tricritical_points.csv'
mf.PANEL_C_ATTRACTORS = data_dir / 'alpha3_fs_42bin_stable_attractors_fs020_090_sample_corrected.csv'
mf.PANEL_C_CURVES = data_dir / 'alpha3_fs_42bin_B056_pseudoarclength_boundaries.csv'
mf.PANEL_C_PITCHFORK_TYPE = data_dir / 'alpha3_fs_42bin_pitchfork_criticality.csv'

required = [
    mf.PANEL_A_GRID, mf.PANEL_A_BOUNDARIES,
    mf.PANEL_B_CACHE, mf.PANEL_B_CURVES, mf.PANEL_B_MACRO,
    mf.PANEL_B_PITCHFORK, mf.PANEL_B_PC,
    mf.PANEL_C_ATTRACTORS, mf.PANEL_C_CURVES, mf.PANEL_C_PITCHFORK_TYPE,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing recomputed data:\n' + '\n'.join(missing))

import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def add_legends(fig, include_bu_nu=False):
    boundary_handles = [
        Line2D([0], [0], color=mf.SADDLE_COLOR, lw=1.8, ls=(0, (2.4, 1.7)), label='Saddle-node'),
        Line2D([0], [0], color=mf.PITCHFORK_COLOR, lw=1.8, ls=(0, (2.4, 1.7)), label='Supercritical PF'),
        Line2D([0], [0], color=mf.SUBCRITICAL_COLOR, lw=1.8, ls=(0, (2.4, 1.7)), label='Subcritical PF'),
        Line2D([0], [0], color=mf.CLASSIFICATION_COLOR, lw=1.8, ls=(0, (2.0, 2.0)), label=r'$B=0.56$'),
    ]
    state_labels = list(mf.THREE_STATE_CLASSES)
    state_handles = [
        Patch(facecolor=mf.THREE_STATE_COLORS[label], edgecolor='black', lw=0.7, label=mf.THREE_STATE_LABELS[label])
        for label in state_labels
    ]
    if include_bu_nu:
        state_handles.append(Patch(facecolor=mf.COEX_BU_NU_COLOR, edgecolor='black', lw=0.7, label='BU–NU coex.'))
    fig.legend(handles=boundary_handles, loc='lower center', bbox_to_anchor=(0.5, 0.115), ncol=2, fontsize=8.5, frameon=False)
    fig.legend(handles=state_handles, loc='lower center', bbox_to_anchor=(0.5, 0.018), ncol=3, fontsize=8.5, frameon=False)

def save_and_show(fig, stem):
    png = figure_dir / f'{stem}.png'
    pdf = figure_dir / f'{stem}.pdf'
    fig.savefig(png, dpi=400, bbox_inches='tight', pad_inches=0.06, facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', pad_inches=0.06, facecolor='white')
    print('Saved:', png)
    print('Saved:', pdf)
    plt.close(fig)
    display(Image(filename=str(png)))

print('Reading data from:', data_dir)
print('Saving figures to:', figure_dir)
print('State mode:', mf.STATE_MODE)

## 1. Panel (a): $(\alpha_2,f_s)$

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 5.0), dpi=180)
mf.draw_panel_a(ax)
mf.style_axis(
    ax, '(a)', r'$\alpha_2$', r'$f_s$',
    (0.0, 0.30), (0.20, 0.90),
    [0.0, 0.1, 0.2, 0.3], [0.2, 0.4, 0.6, 0.9],
    r'$d_c=0.5,\ \alpha_3=0$',
)
add_legends(fig)
fig.subplots_adjust(left=0.16, right=0.97, bottom=0.31, top=0.91)
save_and_show(fig, 'panel_a_alpha2_fs_recomputed')

## 2. Panel (b): $(\alpha_2,d_c)$

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 5.0), dpi=180)
mf.draw_panel_b(ax)
mf.style_axis(
    ax, '(b)', r'$\alpha_2$', r'$d_c$',
    (0.0, 0.30), (0.0, 1.00),
    [0.0, 0.1, 0.2, 0.3], [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    r'$f_s=0.5,\ \alpha_3=0$',
)
add_legends(fig)
fig.subplots_adjust(left=0.16, right=0.97, bottom=0.31, top=0.91)
save_and_show(fig, 'panel_b_alpha2_dc_recomputed')

## 3. Panel (c): $(\alpha_3,f_s)$

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 5.0), dpi=180)
mf.draw_panel_c_replacement(ax)
mf.style_axis(
    ax, '(c)', r'$\alpha_3$', r'$f_s$',
    (0.0, 0.70), (0.20, 0.90),
    [0.0, 0.2, 0.4, 0.6, 0.7], [0.2, 0.4, 0.6, 0.9],
    r'$d_c=0.5,\ \alpha_2=0$',
)
add_legends(fig, include_bu_nu=True)
fig.subplots_adjust(left=0.16, right=0.97, bottom=0.31, top=0.91)
save_and_show(fig, 'panel_c_alpha3_fs_recomputed')

The six output files are written to `figures/recomputed_exact_panels/`. Running one panel cell does not run either of the other panels and does not recalculate any data.